# XGBoost con California Housing 🏠

## Ejercicio de regresión para principiantes

Construiremos un modelo que estima el **valor medio de las viviendas** de distritos de California utilizando información del censo estadounidense de 1990.

El dataset se cargará directamente como un **`DataFrame` de pandas**. Contiene 20,640 filas y 8 variables numéricas.

> Este notebook es educativo. Los valores son históricos y no representan los precios actuales de California.


## Audiencia y objetivos

Este ejercicio es para personas que conocen instrucciones básicas de Python y están comenzando con aprendizaje automático.

Al finalizar podrás:

1. Cargar un dataset como `DataFrame`.
2. Comprender la diferencia entre clasificación y regresión.
3. Separar datos de entrenamiento y prueba.
4. Entrenar un `XGBRegressor`.
5. Interpretar MAE, RMSE y R².
6. Comparar valores reales contra predicciones.


## ¿Qué es una regresión?

En el ejercicio anterior predecíamos una **clase**. Ahora predeciremos un **número continuo**: el valor medio de una vivienda.

XGBoost construye varios árboles pequeños. Cada árbol intenta corregir parte del error cometido por los anteriores. El resultado final es la suma del trabajo de todo el equipo.

**Ruta:** preparar Colab → cargar pandas → explorar → dividir datos → entrenar → predecir → evaluar → experimentar.


## 1. Preparar Google Colab

**Qué haremos:** instalaremos XGBoost `3.3.0` cuando sea necesario.

**Resultado esperado:** un mensaje que confirme la versión disponible.


In [ ]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

VERSION_XGBOOST = "3.3.0"

try:
    version_actual = version("xgboost")
except PackageNotFoundError:
    version_actual = None

if version_actual != VERSION_XGBOOST:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"xgboost=={VERSION_XGBOOST}"],
        check=True,
    )

print(f"XGBoost listo: versión {version('xgboost')}")


**Interpretación:** el entorno está preparado. Ahora importaremos pandas, XGBoost, herramientas para dividir los datos y métricas de regresión.


In [ ]:
import io
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib_config").resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

SEMILLA = 42
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"pandas:  {pd.__version__}")
print(f"XGBoost: {xgb.__version__}")


**Interpretación:** pandas mostrará las tablas; XGBoost entrenará el modelo; scikit-learn proporcionará el dataset, la división y las métricas.

La semilla `42` ayuda a reproducir la misma separación de datos.


## 2. Cargar California Housing como `DataFrame`

El archivo `California_Housing.csv` se proporciona junto con este notebook.

**En Google Colab:** al ejecutar la siguiente celda aparecerá el botón **Elegir archivos**. Selecciona manualmente `California_Housing.csv` desde tu computadora.

Fuera de Colab, la celda buscará el CSV en la misma carpeta que el notebook.

La variable objetivo `ValorMedio` está expresada en unidades de **USD 100,000**. Por ejemplo, `2.50` representa aproximadamente USD 250,000 en el contexto histórico del dataset.

**Resultado esperado:** una tabla con ocho variables y la columna objetivo `ValorMedio`.


In [ ]:
# En Colab se solicita el CSV manualmente; localmente se lee desde la carpeta actual.
try:
    from google.colab import files

    print("Selecciona el archivo California_Housing.csv")
    archivos_subidos = files.upload()
    nombre_archivo = next(iter(archivos_subidos))
    datos = pd.read_csv(io.BytesIO(archivos_subidos[nombre_archivo]))
except ImportError:
    datos = pd.read_csv("California_Housing.csv")

diccionario = pd.DataFrame(
    {
        "variable": [
            "MedInc", "HouseAge", "AveRooms", "AveBedrms",
            "Population", "AveOccup", "Latitude", "Longitude", "ValorMedio"
        ],
        "significado": [
            "Ingreso mediano del distrito",
            "Edad mediana de las viviendas",
            "Promedio de habitaciones por hogar",
            "Promedio de recámaras por hogar",
            "Población del distrito",
            "Promedio de ocupantes por hogar",
            "Latitud",
            "Longitud",
            "Valor medio en unidades de USD 100,000",
        ],
    }
)

display(datos.head(8))
display(diccionario)

columnas_esperadas = {
    "MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population",
    "AveOccup", "Latitude", "Longitude", "ValorMedio"
}
assert columnas_esperadas.issubset(datos.columns), "El CSV no contiene las columnas esperadas."
assert len(datos) == 20640, "El CSV no contiene las 20,640 filas esperadas."

print(f"Archivo cargado: {datos.shape[0]:,} filas y {datos.shape[1]} columnas")
print(f"Valores faltantes: {datos.isna().sum().sum()}")


**Interpretación:** la carga manual produjo un `DataFrame`. Cada fila representa un grupo de viviendas de un distrito censal, no una casa individual. Tenemos 20,640 filas, 8 variables predictoras y una columna objetivo.

No existen valores faltantes en esta versión del dataset, por lo que podemos continuar sin imputación.


## 3. Explorar los datos

**Qué haremos:** calcularemos estadísticas descriptivas y veremos la distribución del valor objetivo.

`describe()` muestra cantidad, promedio, desviación, mínimo, cuartiles y máximo de cada columna.


In [ ]:
resumen = datos.describe().T.round(2)
display(resumen)

plt.figure(figsize=(7, 4))
sns.histplot(datos["ValorMedio"] * 100_000, bins=35, color="steelblue")
plt.title("Distribución del valor medio de las viviendas")
plt.xlabel("Valor medio histórico (USD)")
plt.ylabel("Cantidad de distritos")
plt.show()


**Interpretación:** los distritos tienen valores y características diferentes. La distribución no es perfectamente simétrica y contiene observaciones en el límite superior del dataset.

Los árboles de XGBoost son útiles porque pueden aprender relaciones no lineales sin exigir que todas las variables tengan la misma escala.


## 4. Separar entrenamiento y prueba

- `X` contendrá las ocho variables utilizadas para predecir.
- `y` contendrá `ValorMedio`, la respuesta correcta.
- Usaremos 80% para entrenamiento y 20% para prueba.

**Por qué:** evaluar con datos distintos permite medir cómo funciona el modelo en ejemplos que no utilizó para aprender.


In [ ]:
X = datos.drop(columns="ValorMedio")
y = datos["ValorMedio"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEMILLA,
)

division = pd.DataFrame(
    {
        "conjunto": ["Entrenamiento", "Prueba"],
        "filas": [len(X_train), len(X_test)],
        "variables": [X_train.shape[1], X_test.shape[1]],
    }
)
display(division)


**Interpretación:** el modelo aprenderá únicamente con las filas de entrenamiento. Las 4,128 filas de prueba permanecerán reservadas hasta la evaluación.


## 5. Entrenar `XGBRegressor`

Usaremos tres parámetros principales:

- `n_estimators=250`: número de árboles.
- `max_depth=4`: profundidad máxima de cada árbol.
- `learning_rate=0.05`: tamaño de la corrección aportada por cada árbol.

**Qué haremos:** crearemos el modelo y ejecutaremos `fit` con los datos de entrenamiento.


In [ ]:
modelo_base = xgb.XGBRegressor(
    n_estimators=250,
    max_depth=4,
    learning_rate=0.05,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=SEMILLA,
    n_jobs=2,
)

modelo_base.fit(X_train, y_train)
print("✓ Modelo de regresión entrenado")


**Interpretación:** XGBoost construyó 250 árboles. Cada árbol intentó reducir los errores numéricos acumulados por los anteriores.

El modelo ya puede estimar valores para los distritos de prueba.


## 6. Mostrar predicciones como `DataFrame`

**Qué haremos:** compararemos el valor real y el estimado en dólares históricos. También calcularemos el error absoluto de cada fila.

**Resultado esperado:** una tabla con las primeras doce predicciones.


In [ ]:
predicciones_base = modelo_base.predict(X_test)

resultados = pd.DataFrame(
    {
        "valor_real_USD": (y_test.to_numpy() * 100_000).round(0),
        "prediccion_USD": (predicciones_base * 100_000).round(0),
    }
)
resultados["error_USD"] = (
    resultados["valor_real_USD"] - resultados["prediccion_USD"]
).round(0)
resultados["error_absoluto_USD"] = resultados["error_USD"].abs()

display(resultados.head(12))


**Interpretación:** el error puede ser positivo o negativo según el modelo subestime o sobreestime. El error absoluto ignora la dirección y mide únicamente la distancia respecto al valor real.


## 7. Evaluar el modelo

Usaremos tres métricas:

- **MAE:** error absoluto promedio; menor es mejor.
- **RMSE:** penaliza con más fuerza los errores grandes; menor es mejor.
- **R²:** proporción de variabilidad explicada; cuanto más cerca de 1, mejor.

También graficaremos valores reales contra predicciones. Una predicción perfecta quedaría sobre la línea diagonal.


In [ ]:
mae_base = mean_absolute_error(y_test, predicciones_base)
rmse_base = np.sqrt(mean_squared_error(y_test, predicciones_base))
r2_base = r2_score(y_test, predicciones_base)

metricas_base = pd.DataFrame(
    {
        "métrica": ["MAE", "RMSE", "R²"],
        "valor_original": [mae_base, rmse_base, r2_base],
        "interpretación": [
            f"USD {mae_base * 100_000:,.0f} de error absoluto promedio",
            f"USD {rmse_base * 100_000:,.0f}, penalizando errores grandes",
            f"El modelo explica aproximadamente {r2_base:.1%} de la variación",
        ],
    }
)
metricas_base["valor_original"] = metricas_base["valor_original"].round(4)
display(metricas_base)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, predicciones_base, alpha=0.25, s=15)
limite = [min(y_test.min(), predicciones_base.min()), max(y_test.max(), predicciones_base.max())]
plt.plot(limite, limite, "r--", label="Predicción perfecta")
plt.xlabel("Valor real (unidades de USD 100,000)")
plt.ylabel("Predicción (unidades de USD 100,000)")
plt.title("Valores reales frente a predicciones")
plt.legend()
plt.show()

assert r2_base >= 0.70


**Interpretación:** los puntos cercanos a la diagonal representan buenas predicciones. Las separaciones grandes muestran distritos donde el modelo cometió mayor error.

R² resume la capacidad explicativa, mientras MAE ofrece una medida más fácil de traducir a dólares históricos.


## 8. ¿Qué variables fueron importantes?

XGBoost asigna una importancia a cada variable según cuánto ayudó a construir los árboles.

**Qué haremos:** mostraremos las ocho variables ordenadas. Importancia predictiva no significa causalidad.


In [ ]:
importancia = pd.DataFrame(
    {
        "variable": X_train.columns,
        "importancia": modelo_base.feature_importances_,
    }
).sort_values("importancia", ascending=False)

display(importancia.round(4))

sns.barplot(data=importancia, x="importancia", y="variable")
plt.title("Importancia de variables según XGBoost")
plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.show()


**Interpretación:** una barra más larga indica que la variable ayudó más al modelo. Esto no demuestra que esa variable cause los precios; solamente indica utilidad para predecir dentro de este dataset.

### Beneficios de XGBoost

- Muy buen desempeño con tablas.
- Captura relaciones no lineales e interacciones.
- No necesita escalar todas las columnas.
- Permite regularización e importancia de variables.

### Limitaciones

- Puede sobreajustar con demasiados árboles o profundidad.
- Necesita ajustar parámetros.
- Los datos son históricos y agregados por distrito.
- Una asociación predictiva no demuestra causalidad.


## 9. Ejercicio: cambiar el modelo

Modifica estos valores y vuelve a ejecutar:

- `ARBOLES`: prueba entre 100 y 500.
- `PROFUNDIDAD`: prueba entre 2 y 7.
- `TASA`: prueba 0.03, 0.05, 0.10 o 0.20.

**Pregunta:** ¿el modelo con más árboles siempre mejora lo suficiente para justificar su complejidad?

La celda incluye una configuración válida y una comparación con el modelo base.


In [ ]:
# EJERCICIO: cambia estos tres valores
ARBOLES = 350
PROFUNDIDAD = 5
TASA = 0.05

modelo_experimento = xgb.XGBRegressor(
    n_estimators=ARBOLES,
    max_depth=PROFUNDIDAD,
    learning_rate=TASA,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=SEMILLA,
    n_jobs=2,
)
modelo_experimento.fit(X_train, y_train)
predicciones_experimento = modelo_experimento.predict(X_test)

mae_experimento = mean_absolute_error(y_test, predicciones_experimento)
rmse_experimento = np.sqrt(mean_squared_error(y_test, predicciones_experimento))
r2_experimento = r2_score(y_test, predicciones_experimento)

comparacion = pd.DataFrame(
    {
        "modelo": ["Base", "Experimento"],
        "árboles": [250, ARBOLES],
        "profundidad": [4, PROFUNDIDAD],
        "tasa": [0.05, TASA],
        "MAE_USD": [mae_base * 100_000, mae_experimento * 100_000],
        "RMSE_USD": [rmse_base * 100_000, rmse_experimento * 100_000],
        "R²": [r2_base, r2_experimento],
    }
).round(3)
display(comparacion)

if rmse_experimento < rmse_base:
    print("Solución: el experimento reduce RMSE; revisa si la mejora justifica usar más árboles.")
else:
    print("Solución: el modelo base tiene menor RMSE y además es más sencillo.")


**Interpretación de la solución:** el mejor modelo no es necesariamente el más grande. Buscamos MAE y RMSE bajos, R² alto y una complejidad razonable.

Reducir la tasa de aprendizaje hace más pequeña la contribución de cada árbol, por lo que normalmente requiere utilizar más árboles.


## Conclusión

Aprendimos a:

- Subir manualmente un CSV a Google Colab y cargarlo como `DataFrame`.
- Comprender un problema de regresión.
- Separar entrenamiento y prueba.
- Entrenar un `XGBRegressor`.
- Mostrar predicciones y errores como tablas.
- Evaluar con MAE, RMSE y R².
- Comparar dos configuraciones.

**Error común:** interpretar estos valores como precios actuales. El dataset se deriva del censo de 1990 y el objetivo está expresado en unidades de USD 100,000.

### Fuentes oficiales

- [California Housing en scikit-learn](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset)
- [`fetch_california_housing`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html)
- [Interfaz scikit-learn de XGBoost](https://xgboost.readthedocs.io/en/stable/python/sklearn_estimator.html)
